# Tutorial

## Contracts

In [1]:
import z3
from agsheaf import Contract, Relation, ContractSheaf
from agsheaf.contracts import top, bot

Define variables

In [2]:
[p1, p2, p3, p4] = z3.Bools("p1 p2 p3 p4")

Define two A/G contracts

In [3]:
c1 = Contract(z3.And(p1, p2),p3)
c2 = Contract(p4, z3.Or(p1, p2))

print(c1)
print(c2)

Contract(
  A: And(p1, p2)
  G: Or(p3, Not(And(p1, p2)))
)
Contract(
  A: p4
  G: Or(p1, p2, Not(p4))
)


Compute the meets and joints of contracts

In [4]:
c_meet = c1.meet(c2)
c_join = c1.join(c2)

print(c_meet)
print(c_join)

Contract(
  A: Or(p4, And(p1, p2))
  G: Or(Not(Or(p4, And(p1, p2))),
   And(Or(p3, Not(And(p1, p2))), Or(p1, p2, Not(p4))))
)
Contract(
  A: And(p1, p2, p4)
  G: Or(p1,
   p2,
   p3,
   Not(p4),
   Not(And(p1, p2, p4)),
   Not(And(p1, p2)))
)


Define `Relations`

* inference of `p4` from `p3` by Agent 1
* inference of `p3` from `p4` by Agent 2

In [5]:
rel_1_to_2 = Relation(z3.Implies(p3, p4), [p1,p2,p3],[p1,p2,p4])
rel_2_to_1 = Relation(z3.Implies(p4, p3), [p1,p2,p4],[p1,p2,p3])

In [6]:
rel_1_to_2

Relation(
  R: Or(p4, Not(p3))
  X: [p1, p2, p3]
  Y: [p1, p2, p4]
)

In [7]:
rel_2_to_1

Relation(
  R: Or(p3, Not(p4))
  X: [p1, p2, p4]
  Y: [p1, p2, p3]
)

Compute $\mathrm{Ran}$

In [8]:
c1

Contract(
  A: And(p1, p2)
  G: Or(p3, Not(And(p1, p2)))
)

In [9]:
c1.ran(rel_1_to_2)

Contract(
  A: Exists(p3, And(Or(p4, Not(p3)), p1, p2))
  G: Or(Not(Exists(p3, And(Or(p4, Not(p3)), p1, p2))),
   Not(p1),
   Not(p2))
)

In [10]:
c2

Contract(
  A: p4
  G: Or(p1, p2, Not(p4))
)

In [11]:
c2.ran(rel_2_to_1)

Contract(
  A: p3
  G: Or(p1, p2, Not(p3))
)

Compute $\mathrm{Lan}$

In [12]:
c1.lan(rel_1_to_2)

Contract(
  A: ForAll(p3, Or(Not(Or(p4, Not(p3))), And(p1, p2)))
  G: Or(Not(ForAll(p3, Or(Not(Or(p4, Not(p3))), And(p1, p2)))),
   Exists(p3,
          And(Or(p4, Not(p3)), Or(Not(And(p1, p2)), p3))))
)

In [13]:
c2.lan(rel_2_to_1)

Contract(
  A: False
  G: True
)

## Contract sheaves

In [14]:
import z3
from agsheaf import Contract, Relation, ContractSheaf
from agsheaf.contracts import top, bot
import networkx as nx

Program a contract sheaf over a graph with agents (nodes) and a single interface (edge).

A `ContractSheaf` restriction carries an agent onto the **shared edge stalk**, not directly onto its neighbour, so both directions of an interface must land in the same alphabet. Here agent 1 speaks about $\{p_1, p_2, p_3\}$ and agent 2 about $\{p_1, p_2, p_4\}$, and the edge between them is $\{p_1, p_2, e\}$ where $e$ couples $p_3$ to $p_4$ — expressing the mutual inference of the previous section.

Note that $p_1$ and $p_2$ appear in *both* alphabets. They are coordinates the two agents hold in common, and the Kan maps leave them free rather than quantifying over them.

In [15]:
[p1, p2, p3, p4] = z3.Bools("p1 p2 p3 p4")
e = z3.Bool("e")

F = ContractSheaf()
F.add_node(1, c=Contract(z3.And(p1, p2), p3))
F.add_node(2, c=Contract(p4, z3.Or(p1, p2)))
F.add_interface(1, 2,
                Relation(e == p3, [p1, p2, p3], [p1, p2, e]),
                Relation(e == p4, [p1, p2, p4], [p1, p2, e])
)

Show the contracts (in saturated form)

In [16]:
F.contract(1)

Contract(
  A: And(p1, p2)
  G: Or(p3, Not(And(p1, p2)))
)

In [17]:
F.contract(2)

Contract(
  A: p4
  G: Or(p1, p2, Not(p4))
)

Show the relations between agents and interfaces

In [18]:
F.restriction(1,2)

Relation(
  R: e == p3
  X: [p1, p2, p3]
  Y: [p1, p2, e]
)

In [19]:
F.restriction(2,1)

Relation(
  R: e == p4
  X: [p1, p2, p4]
  Y: [p1, p2, e]
)

The initial contract assignment is stored. In this case, it is NOT a section.

In [20]:
F.initial(1)

Contract(
  A: And(p1, p2)
  G: Or(p3, Not(And(p1, p2)))
)

In [21]:
F.initial(2)

Contract(
  A: p4
  G: Or(p1, p2, Not(p4))
)

In [22]:
F.is_section()

False

## The harmonic flow

The two agents disagree, so we diffuse. Each step replaces every agent's contract with the meet of its own and everything its neighbours can say about it — $ x \gets L x \wedge x$, the heat flow of Riess and Ghrist (2022), Eq. (6).

`converge` runs this to a fixed point. With `verify=True` it also checks, against a frozen assignment, that the point it stopped at really is a suffix point of the Laplacian.

In [23]:
steps = F.converge(verify=True)
print(f"settled after {steps} step(s)")
print("is_section:", F.is_section())

settled after 1 step(s)
is_section: True


In [25]:
F.initial(1)

Contract(
  A: And(p1, p2)
  G: Or(p3, Not(And(p1, p2)))
)

In [26]:
F.contract(1)

Contract(
  A: Or(p3, And(p1, p2))
  G: Or(Not(Or(p3, And(p1, p2))),
   And(Or(Not(Or(p3, And(p1, p2))),
          And(Or(p3, Not(And(p1, p2))), Or(p1, p2, Not(p3)))),
       Or(Not(Or(p3, And(p1, p2))),
          And(Or(p1, p2, Not(p3)),
              Or(Not(Or(p3, And(p1, p2))),
                 And(Or(p3, Not(And(p1, p2))),
                     Or(p1, p2, Not(p3))))))))
)

By the Hodge–Lawvere Theorem (Ghrist et al. 2026, Thm. 6.10) the suffix points of the Laplacian *are* the global sections, so over this Boolean base `is_suffix()` and `is_section()` agree — the flow has computed a consistent joint assignment.

`initial` still holds the assignment as given, so the two can be compared.

In [27]:
print("is_suffix:", F.is_suffix())

for i in F.nodes():
    print(f"agent {i}: refines its initial contract:",
          F.contract(i).refines(F.initial(i)))

is_suffix: True
agent 1: refines its initial contract: True
agent 2: refines its initial contract: True


## Asynchronous updates

Nothing above required the agents to act in lockstep. A *firing sequence* $\tau$ selects which agents broadcast at each step (Riess and Ghrist 2022, Def. 5), and their Theorem 1 says that as long as every agent fires infinitely often — *liveness* — the flow reaches the same answer whatever the schedule.

`round_robin` fires one agent at a time, the maximally asynchronous choice.

In [29]:
from agsheaf.sheaf import round_robin

F.reset()                       # back to the assignment as given
print("after reset, is_section:", F.is_section())

steps = F.converge(firing_sequence=lambda: round_robin([1, 2]))
print(f"settled after {steps} asynchronous step(s)")
print("is_section:", F.is_section())

after reset, is_section: False
settled after 2 asynchronous step(s)
is_section: True
